# Driver Cellphone Use YOLO Training Template

This notebook uses the attached YOLO-format dataset, validates it, fine-tunes YOLO on Kaggle GPU, evaluates the trained model on validation and test splits, benchmarks inference, and zips useful artifacts for download.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/tinhvu11235/homeobjects-yolo-detector.git'
PROJECT_DIR = Path('/kaggle/working/homeobjects-yolo-detector')
SRC_DIR = PROJECT_DIR / 'src'
DATASET_CHECK = SRC_DIR / 'dataset_check.py'
TRAIN_SCRIPT = SRC_DIR / 'train.py'
EVALUATE_SCRIPT = SRC_DIR / 'evaluate.py'
BENCHMARK_SCRIPT = SRC_DIR / 'benchmark.py'

if PROJECT_DIR.exists():
    print(f'Repository already exists: {PROJECT_DIR}')
else:
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
print('Working directory:', Path.cwd())
print('Python:', sys.version)

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_DIR / 'requirements.txt')], check=True)

import torch
import ultralytics

print('Ultralytics:', ultralytics.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator before training.')
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import yaml

INPUT_ROOT = Path('/kaggle/input')
if not INPUT_ROOT.exists():
    raise FileNotFoundError('/kaggle/input does not exist. Attach the converted YOLO dataset first.')

print('Attached Kaggle datasets:')
for child in sorted(INPUT_ROOT.iterdir()):
    print('-', child)

yaml_candidates = sorted(set(INPUT_ROOT.rglob('data.yaml')) | set(INPUT_ROOT.rglob('*.yaml')) | set(INPUT_ROOT.rglob('*.yml')))
if not yaml_candidates:
    raise FileNotFoundError('No data.yaml/.yaml file found. Attach the converted YOLO dataset, not the raw COCO export.')

EXPECTED_CLASSES = {'Cellphone-in-drivers', 'driver', 'phone', 'wheel'}

def load_yaml(path):
    with path.open('r', encoding='utf-8') as file:
        return yaml.safe_load(file) or {}

def class_names(config):
    names = config.get('names', {})
    if isinstance(names, dict):
        return {str(value) for value in names.values()}
    if isinstance(names, list):
        return {str(value) for value in names}
    return set()

def yaml_score(path):
    text = str(path).lower()
    score = 0
    if path.name.lower() == 'data.yaml':
        score += 10
    for keyword in ['cellphone', 'phone', 'driver']:
        if keyword in text:
            score += 10
    try:
        config = load_yaml(path)
    except Exception:
        return score
    if {'train', 'val', 'names'}.issubset(config.keys()):
        score += 20
    if EXPECTED_CLASSES.issubset(class_names(config)):
        score += 100
    return score

DATA_YAML = max(yaml_candidates, key=yaml_score)
with DATA_YAML.open('r', encoding='utf-8') as file:
    DATA_CONFIG = yaml.safe_load(file) or {}

found_classes = class_names(DATA_CONFIG)
missing_classes = sorted(EXPECTED_CLASSES - found_classes)
if missing_classes:
    raise ValueError(f'Attached YOLO data.yaml is missing expected classes: {missing_classes}. Using: {DATA_YAML}')

def split_exists_at_root(config, root, split_name):
    value = config.get(split_name)
    items = value if isinstance(value, list) else [value]
    for item in items:
        if item in (None, ''):
            continue
        path = Path(str(item))
        candidate = path if path.is_absolute() else (root / path).resolve()
        if candidate.exists():
            return True
    return False

def dataset_root_from_config(config, yaml_path):
    candidates = []
    raw_root = config.get('path')
    if raw_root not in (None, ''):
        root = Path(str(raw_root))
        candidates.append(root if root.is_absolute() else (yaml_path.parent / root).resolve())
    candidates.append(yaml_path.parent)

    for candidate in candidates:
        if any(split_exists_at_root(config, candidate, split_name) for split_name in ('train', 'val', 'test')):
            return candidate.resolve()
    return candidates[0].resolve()

def resolve_split_path(config, yaml_path, split_name):
    root = dataset_root_from_config(config, yaml_path)
    value = config.get(split_name)
    items = value if isinstance(value, list) else [value]
    for item in items:
        if item in (None, ''):
            continue
        path = Path(str(item))
        path = path if path.is_absolute() else (root / path).resolve()
        if path.exists():
            return path
    return None

ORIGINAL_DATA_YAML = DATA_YAML
DATASET_ROOT = dataset_root_from_config(DATA_CONFIG, ORIGINAL_DATA_YAML)
PATCHED_DATA_CONFIG = dict(DATA_CONFIG)
PATCHED_DATA_CONFIG['path'] = str(DATASET_ROOT)
PATCHED_DATA_YAML = PROJECT_DIR / 'outputs' / 'cellphone_drivers_kaggle_data.yaml'
PATCHED_DATA_YAML.parent.mkdir(parents=True, exist_ok=True)
with PATCHED_DATA_YAML.open('w', encoding='utf-8') as file:
    yaml.safe_dump(PATCHED_DATA_CONFIG, file, sort_keys=False, allow_unicode=True)

DATA_YAML = PATCHED_DATA_YAML
DATA_CONFIG = PATCHED_DATA_CONFIG
VAL_IMAGES = resolve_split_path(DATA_CONFIG, DATA_YAML, 'val')
if VAL_IMAGES is None:
    raise FileNotFoundError(f'Could not resolve validation images from {DATA_YAML}')

print('Original dataset YAML:', ORIGINAL_DATA_YAML)
print('Patched training YAML:', DATA_YAML)
print('Dataset root:', DATASET_ROOT)
print('Validation images:', VAL_IMAGES)
print('YAML keys:', sorted(DATA_CONFIG.keys()))
print('Classes:', DATA_CONFIG.get('names'))

In [ ]:
!python "{DATASET_CHECK}" --data "{DATA_YAML}" --output "{PROJECT_DIR / 'outputs' / 'dataset_report_cellphone_drivers.json'}"

In [ ]:
!python "{TRAIN_SCRIPT}" \
  --data "{DATA_YAML}" \
  --model yolo11s.pt \
  --epochs 80 \
  --imgsz 640 \
  --batch 16 \
  --device 0 \
  --project "{PROJECT_DIR / 'runs'}" \
  --name cellphone_drivers_yolo

In [ ]:
BEST_PT = PROJECT_DIR / 'runs' / 'cellphone_drivers_yolo' / 'weights' / 'best.pt'
print('best.pt:', BEST_PT, 'exists:', BEST_PT.exists())
if not BEST_PT.exists():
    raise FileNotFoundError('Training did not produce best.pt.')

# Required post-training evaluation on validation split.
!python "{EVALUATE_SCRIPT}" --weights "{BEST_PT}" --data "{DATA_YAML}" --imgsz 640 --device 0 --split val --output "{PROJECT_DIR / 'outputs' / 'eval_summary_val.json'}"

# Additional held-out test evaluation.
!python "{EVALUATE_SCRIPT}" --weights "{BEST_PT}" --data "{DATA_YAML}" --imgsz 640 --device 0 --split test --output "{PROJECT_DIR / 'outputs' / 'eval_summary_test.json'}"

In [ ]:
if VAL_IMAGES is None or not VAL_IMAGES.exists():
    raise FileNotFoundError(f'Validation image folder not found: {VAL_IMAGES}')

!python "{BENCHMARK_SCRIPT}" \
  --weights "{BEST_PT}" \
  --source "{VAL_IMAGES}" \
  --imgsz 640 \
  --device 0 \
  --warmup 20 \
  --runs 500 \
  --save-json "{PROJECT_DIR / 'outputs' / 'benchmark_summary.json'}" \
  --save-csv "{PROJECT_DIR / 'outputs' / 'benchmark_results.csv'}"

In [ ]:
import zipfile

artifact_paths = [
    PROJECT_DIR / 'runs' / 'cellphone_drivers_yolo' / 'weights' / 'best.pt',
    PROJECT_DIR / 'runs' / 'cellphone_drivers_yolo' / 'weights' / 'last.pt',
    PROJECT_DIR / 'runs' / 'cellphone_drivers_yolo' / 'results.png',
    PROJECT_DIR / 'runs' / 'cellphone_drivers_yolo' / 'confusion_matrix.png',
    PROJECT_DIR / 'runs' / 'cellphone_drivers_yolo' / 'PR_curve.png',
    PROJECT_DIR / 'runs' / 'cellphone_drivers_yolo' / 'F1_curve.png',
    PROJECT_DIR / 'runs' / 'cellphone_drivers_yolo' / 'args.yaml',
    PROJECT_DIR / 'outputs' / 'dataset_report_cellphone_drivers.json',
    PROJECT_DIR / 'outputs' / 'eval_summary_val.json',
    PROJECT_DIR / 'outputs' / 'eval_summary_test.json',
    PROJECT_DIR / 'outputs' / 'benchmark_summary.json',
    PROJECT_DIR / 'outputs' / 'benchmark_results.csv',
]

zip_path = PROJECT_DIR / 'cellphone_drivers_yolo_artifacts.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zip_file:
    for path in artifact_paths:
        if path.exists():
            zip_file.write(path, arcname=str(path.relative_to(PROJECT_DIR)))
        else:
            print('Missing artifact, skipped:', path)

print('Created:', zip_path.resolve())
print('Download best.pt and place it in models/best.pt for Hugging Face deployment.')

## Deployment Step

After training and evaluation, download `runs/cellphone_drivers_yolo/weights/best.pt` from Kaggle output and place it into `models/best.pt` in the repository. Hugging Face Spaces will run `app.py` for CPU inference only.